In [ ]:
from memory import Memory
from planner import plan_steps
from retrieval import retrieve
from tools import calculate_emi_tool, escalate_tool
from extractor import extract_loan_details

memory = Memory()


def agent_v6(user_input):
    context = memory.get_context()
    steps = plan_steps(user_input)

    response_parts = []

    for step in steps:

        if step == "extract_loan_details":
            details = extract_loan_details(user_input)
            if not details:
                return "Please provide loan amount, interest rate, and tenure."

        elif step == "calculate_emi":
            emi = calculate_emi_tool(
                details["principal"],
                details["rate"],
                details["tenure"]
            )
            response_parts.append(emi)

        elif step == "retrieve_knowledge":
            docs = retrieve(user_input)
            response_parts.append("\n".join(docs))

        elif step == "escalate":
            return escalate_tool("Fraud suspected")

        elif step == "fallback":
            response_parts.append("Let me connect you to support.")

    final_response = "\n".join(response_parts)

    # Save memory
    memory.add_to_history(user_input, final_response)

    return final_response


if __name__ == "__main__":
    while True:
        user_input = input("You: ")

        if user_input == "reset":
            memory.reset()
            print("Memory cleared.")
            continue

        print(agent_v6(user_input))